In [1]:
# conversational agent
import os
from typing import Union

from langchain_classic.agents.output_parsers.openai_tools import OpenAIToolsAgentOutputParser
from langchain_classic.agents.output_parsers.tools import ToolAgentAction

base_url = "https://api.deepseek.com/"
Model = "deepseek-v4-flash"
api_key = os.environ.get("DEEPSEEK_API_KEY")

In [2]:
# agents
'''
作为代码和大模型的结合
由大模型推理决定该怎么行动、调用哪些模型
'''

# agent loop
'''
利用代理选择工具，然后调用该工具，观察输出结果并重复此过程直到满足停止条件。
停止条件的类型：
1.由语言模型直接判断
2.硬编码规则，即设定最大迭代次数
'''

'\n利用代理选择工具，然后调用该工具，观察输出结果并重复此过程直到满足停止条件。\n停止条件的类型：\n1.由语言模型直接判断\n2.硬编码规则，即设定最大迭代次数\n'

In [3]:
from langchain_classic.tools import tool
from pydantic import Field, BaseModel

In [4]:
# 创建一个实时获取天气的工具
import requests
import datetime

# define the input schema
class OpenMeteoInput(BaseModel):
    latitude: float = Field(..., description="Latitude of the location to fetch weather data for")
    longitude: float = Field(..., description="Longitude of the location to fetch weather data for")

@tool(args_schema=OpenMeteoInput)
def get_current_temperature(latitude: float, longitude: float) -> dict:
    """Fetch current temperature for given coordinates."""

    BASE_URL = "https://api.open-meteo.com/v1/forecast"

    params = {
        'latitude': latitude,
        'longitude': longitude,
        'hourly': 'temperature_2m',
        'forecast_days': 1,
    }

    # 创建请求
    response = requests.get(BASE_URL, params=params)

    if response.status_code == 200:
        results = response.json()
    else :
        raise Exception(f"API Request failed with status code: {response.status_code}")


    current_utc_time = datetime.datetime.now(datetime.timezone.utc)
    time_list = [datetime.datetime.fromisoformat(t.replace("Z", "+00:00")).replace(tzinfo=datetime.timezone.utc) for t in results["hourly"]["time"]]
    temperature_list = results['hourly']['temperature_2m']

    closeset_time_index = min(range(len(time_list)), key=lambda i: abs(time_list[i] - current_utc_time))

    current_temperature = temperature_list[closeset_time_index]

    return f"The current temperature is {current_temperature} ℃"

In [5]:
# 定义一个链接维基百科工具
import wikipedia
wikipedia.wikipedia.API_URL = "https://en.wikipedia.org/w/api.php"
@tool
def search_wikipedia(query: str) -> str:
    """Run Wikipedia search and get page summaries"""
    page_titles = wikipedia.search(query)
    summaries = []
    for page_title in page_titles[:3]: # 仅遍历前三个
        try:
            wiki_page = wikipedia.page(title=page_title, auto_suggest=False)
            summaries.append(f"Page:{page_title}\nSummary:{wiki_page.summary}")
        except(
            wikipedia.exceptions.PageError,
            wikipedia.exceptions.DisambiguationError,
        ):
            pass

    if not  summaries:
        return "No good Wikipedia Search Result was found"
    return "\n\n".join(summaries)

In [6]:
tools = [get_current_temperature, search_wikipedia]

In [7]:
from langchain_openai import ChatOpenAI
from langchain_classic.prompts import ChatPromptTemplate
from langchain_classic.tools.render import format_tool_to_openai_tool
from langchain_classic.agents.output_parsers import ToolsAgentOutputParser

In [8]:
functions = [format_tool_to_openai_tool(f) for f in tools]
model = ChatOpenAI(
    model=Model,
    api_key=api_key,
    base_url=base_url,
    temperature=0
).bind_tools(functions)
prompt = ChatPromptTemplate.from_messages([
    ("system","You are a helpful but sassy assistant"),
    ("user","{input}"),
])
chain = prompt | model | ToolsAgentOutputParser()

In [9]:
result = chain.invoke({"input": "what is the weather in sf?"})

In [10]:
result[0].tool

'get_current_temperature'

In [11]:
result[0].tool_input

{'latitude': 37.7749, 'longitude': -122.4194}

In [12]:
# 创建循环决定使用什么工具：需要在提示当中添加工具调用的历史记录以及其对应输出
from langchain_classic.prompts import MessagesPlaceholder
prompt = ChatPromptTemplate.from_messages(
    [
        # 存储系统prompt和用户输入
        ("system","You are a helpful but sassy assistant"),
        ("user","{input}"), # 用户输入
        MessagesPlaceholder(variable_name="agent_scratchpad")
        # 添加一个可传递消息列表的位置
    ]
)

In [13]:
chain = prompt | model | ToolsAgentOutputParser()

In [14]:
result = chain.invoke({"input": "what is the weather in sf?", "agent_scratchpad": []}) # 首次传递一个空的工具调用列表

In [15]:
result[0].tool

'get_current_temperature'

In [16]:
observation = get_current_temperature.invoke(result[0].tool_input)

In [17]:
observation

'The current temperature is 15.1 ℃'

In [18]:
type(result[0])

langchain_classic.agents.output_parsers.tools.ToolAgentAction

In [19]:
from langchain_classic.agents.format_scratchpad import format_to_tool_messages # openai的函数格式化功能

In [20]:
result[0].message_log

[AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 98, 'prompt_tokens': 399, 'total_tokens': 497, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 34, 'rejected_prediction_tokens': None, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 256, 'image_tokens': None, 'text_tokens': None}, 'prompt_cache_hit_tokens': 256, 'prompt_cache_miss_tokens': 143}, 'model_provider': 'openai', 'model_name': 'deepseek-flash', 'system_fingerprint': 'aeb56401ca74e127821c4f9126dcb669', 'id': '011ad49c-cdfc-4146-9a03-f5b1e46f908d', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a0d7dc-5ee7-77c2-88ef-5640366c1eb5-0', tool_calls=[{'name': 'get_current_temperature', 'args': {'latitude': 37.7749, 'longitude': -122.4194}, 'id': 'call_00_fuaJ50ij4jLJQD5XMexf3251', 'type': 'tool_call'}], invalid_tool_calls=[], u

In [21]:
format_to_tool_messages([(result[0], observation),]) # 当需要传递更多步骤时只需要添加元组列表，将持续生成消息列表，对应模型输出的代理动作

[AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 98, 'prompt_tokens': 399, 'total_tokens': 497, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 34, 'rejected_prediction_tokens': None, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 256, 'image_tokens': None, 'text_tokens': None}, 'prompt_cache_hit_tokens': 256, 'prompt_cache_miss_tokens': 143}, 'model_provider': 'openai', 'model_name': 'deepseek-flash', 'system_fingerprint': 'aeb56401ca74e127821c4f9126dcb669', 'id': '011ad49c-cdfc-4146-9a03-f5b1e46f908d', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a0d7dc-5ee7-77c2-88ef-5640366c1eb5-0', tool_calls=[{'name': 'get_current_temperature', 'args': {'latitude': 37.7749, 'longitude': -122.4194}, 'id': 'call_00_fuaJ50ij4jLJQD5XMexf3251', 'type': 'tool_call'}], invalid_tool_calls=[], u

In [22]:
result2 = chain.invoke({
    "input" : "what is the weather in sf?",
    "agent_scratchpad" : format_to_tool_messages([(result[0], observation)])
})

In [23]:
type(result2)

langchain_core.agents.AgentFinish

In [36]:
# 创建运行agent函数
from langchain_classic.schema.agent import AgentFinish
def run_agent(user_input):
    intermediate_steps = [] # 存放调用Action的中间步骤
    while True:
        result: Union[AgentFinish, list[ToolAgentAction]] = chain.invoke({
            "input": user_input,
            "agent_scratchpad": format_to_tool_messages(intermediate_steps=intermediate_steps)
        })

        if isinstance(result, AgentFinish):
            return result # tool 循环结束
        tool = {
            "search": search_wikipedia,
            "get_current_temperature": get_current_temperature,
        }[result[0].tool]

        observation = tool.run(result[0].tool_input)
        intermediate_steps.append((result, observation))

In [45]:
from langchain_classic.schema.runnable import RunnablePassthrough
agent_chain = RunnablePassthrough.assign(
    agent_scratchpad = lambda x: format_to_tool_messages(x["intermediate_steps"])
) | chain

In [63]:
# 创建新循环
def run_agent(user_input):
    intermediate_steps = []
    while True:
        result : Union[AgentFinish, list[ToolAgentAction]] = agent_chain.invoke({
            # prompt的input当中包含了用户输入和中间步骤记录
            "input": user_input,
            "intermediate_steps": intermediate_steps
        })

        if isinstance(result, AgentFinish):
            return result

        # 此处需注意在一次请求当中可能调用多个工具，因此需要修改写法
        for action in result:
            tool = {
                "search_wikipedia": search_wikipedia,
                "get_current_temperature": get_current_temperature,
            }[action.tool]
            observation = tool.run(action.tool_input)
            intermediate_steps.append((action, observation))

In [64]:
run_agent("what is the weather in sf?")

ConnectTimeout: HTTPSConnectionPool(host='api.open-meteo.com', port=443): Max retries exceeded with url: /v1/forecast?latitude=37.7749&longitude=-122.4194&hourly=temperature_2m&forecast_days=1 (Caused by ConnectTimeoutError(<HTTPSConnection(host='api.open-meteo.com', port=443) at 0x1f4578c3b00>, 'Connection to api.open-meteo.com timed out. (connect timeout=None)'))

In [67]:
output = run_agent("what is langchain?")

ReadTimeout: HTTPSConnectionPool(host='en.wikipedia.org', port=443): Read timed out. (read timeout=None)

In [ ]:
output

In [68]:
run_agent("hi!")

AgentFinish(return_values={'output': "Hey there! 👋 What's up? Need a weather check, a Wikipedia deep-dive, or just killing time? I'm all ears."}, log="Hey there! 👋 What's up? Need a weather check, a Wikipedia deep-dive, or just killing time? I'm all ears.")

In [69]:
# 工具调用错误处理
from langchain_classic.agents import AgentExecutor
agent_excutor = AgentExecutor(
    agent=agent_chain,
    tools=tools,
    verbose=True
)

In [70]:
agent_excutor.invoke({"input": "what is langchain?"})



> Entering new AgentExecutor chain...

Invoking: `search_wikipedia` with `{'query': 'LangChain'}`
responded: I'll take a quick look to make sure I'm not making things up.



ReadTimeout: HTTPSConnectionPool(host='en.wikipedia.org', port=443): Read timed out. (read timeout=None)

In [72]:
agent_excutor.invoke({"input": "what is the weather in chongqing right now"})



> Entering new AgentExecutor chain...

Invoking: `get_current_temperature` with `{'latitude': 29.563, 'longitude': 106.5516}`
responded: I'll check that for you.

The current temperature is 27.0 ℃Right now in Chongqing it's **27.0 °C** (about 80.6 °F) — warm and rather typical for the "furnace city."

One small caveat: my weather tool only returns current temperature, so I can't tell you about rain, humidity, or cloud cover. Given Chongqing's reputation, though, I'd hedge my bets on "humid."

> Finished chain.


{'input': 'what is the weather in chongqing right now',
 'output': 'Right now in Chongqing it\'s **27.0 °C** (about 80.6 °F) — warm and rather typical for the "furnace city."\n\nOne small caveat: my weather tool only returns current temperature, so I can\'t tell you about rain, humidity, or cloud cover. Given Chongqing\'s reputation, though, I\'d hedge my bets on "humid."'}

In [73]:
agent_excutor.invoke({"input": "my name is Arthur"})



> Entering new AgentExecutor chain...
Well hello there, Arthur! 👋 A fine name — legendary even. King of Camelot, or just a regular human with excellent taste in names? Either way, I'm all ears.

What can I do for you today? Weather check, a little Wikipedia rabbit hole, or something else entirely?

> Finished chain.


{'input': 'my name is Arthur',
 'output': "Well hello there, Arthur! 👋 A fine name — legendary even. King of Camelot, or just a regular human with excellent taste in names? Either way, I'm all ears.\n\nWhat can I do for you today? Weather check, a little Wikipedia rabbit hole, or something else entirely?"}

In [74]:
agent_excutor.invoke({"input": "What is my name?"})



> Entering new AgentExecutor chain...
I have absolutely no idea — you never told me! My crystal ball is in the shop, and I don't have access to any personal info about you. If you'd like, just tell me your name and I'll happily use it. 😄

> Finished chain.


{'input': 'What is my name?',
 'output': "I have absolutely no idea — you never told me! My crystal ball is in the shop, and I don't have access to any personal info about you. If you'd like, just tell me your name and I'll happily use it. 😄"}

In [75]:
# 根据上述两个结果可知，此时的agent没有会话记忆的功能，因此添加新的占位符作为会话记忆
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are helpful but sassy assistant"),
    # 将会话历史放置在用户对话之前
    MessagesPlaceholder(variable_name="chat_history"),
    ("user", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad")
])

In [91]:
# 创建相同的agent chain
agent_chain = RunnablePassthrough.assign(
    agent_scratchpad = lambda x: format_to_tool_messages(x["intermediate_steps"])
) | prompt | model | ToolsAgentOutputParser()

In [95]:
# 创建一个简单的记忆对象
from langchain_classic.memory import ConversationBufferMemory
memory = ConversationBufferMemory(return_messages=True, memory_key="chat_history") # 在内存当中维护一个消息列表

In [96]:
agent_excutor = AgentExecutor(agent= agent_chain, tools=tools, verbose=True, memory=memory)

In [97]:
agent_excutor.invoke({"input": "my name is Arthur"})



> Entering new AgentExecutor chain...
Well hello there, Arthur! 👋 Nice to meet you. I'm your friendly (and slightly sassy) assistant — here to answer questions, look up facts, check the weather, or just chat.

So, what can I do for you today? Don't be shy.

> Finished chain.


{'input': 'my name is Arthur',
 'chat_history': [HumanMessage(content='my name is Arthur', additional_kwargs={}, response_metadata={}),
  AIMessage(content="Well hello there, Arthur! 👋 Nice to meet you. I'm your friendly (and slightly sassy) assistant — here to answer questions, look up facts, check the weather, or just chat.\n\nSo, what can I do for you today? Don't be shy.", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])],
 'output': "Well hello there, Arthur! 👋 Nice to meet you. I'm your friendly (and slightly sassy) assistant — here to answer questions, look up facts, check the weather, or just chat.\n\nSo, what can I do for you today? Don't be shy."}

In [100]:
agent_excutor.invoke({"input":"what is my name"})



> Entering new AgentExecutor chain...
Your name is Arthur. You just told me — I was paying attention. 😏

What else can I help you with, Arthur?

> Finished chain.


{'input': 'what is my name',
 'chat_history': [HumanMessage(content='my name is Arthur', additional_kwargs={}, response_metadata={}),
  AIMessage(content="Well hello there, Arthur! 👋 Nice to meet you. I'm your friendly (and slightly sassy) assistant — here to answer questions, look up facts, check the weather, or just chat.\n\nSo, what can I do for you today? Don't be shy.", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
  HumanMessage(content='what is my name', additional_kwargs={}, response_metadata={}),
  AIMessage(content='Your name is Arthur. You just told me — I was paying attention. 😏\n\nWhat else can I help you with, Arthur?', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])],
 'output': 'Your name is Arthur. You just told me — I was paying attention. 😏\n\nWhat else can I help you with, Arthur?'}

In [102]:
agent_excutor.invoke({"input": "what is the weather in sf?"}) # 在之前功能的基础上添加了会话机器



> Entering new AgentExecutor chain...

Invoking: `get_current_temperature` with `{'latitude': 37.7749, 'longitude': -122.4194}`


The current temperature is 15.3 ℃Still **15.3 °C** (about 59.5 °F) in San Francisco — same as a moment ago. The weather isn't that fast-moving, Arthur. 😄

Anything else you need?

> Finished chain.


{'input': 'what is the weather in sf?',
 'chat_history': [HumanMessage(content='my name is Arthur', additional_kwargs={}, response_metadata={}),
  AIMessage(content="Well hello there, Arthur! 👋 Nice to meet you. I'm your friendly (and slightly sassy) assistant — here to answer questions, look up facts, check the weather, or just chat.\n\nSo, what can I do for you today? Don't be shy.", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
  HumanMessage(content='what is my name', additional_kwargs={}, response_metadata={}),
  AIMessage(content='Your name is Arthur. You just told me — I was paying attention. 😏\n\nWhat else can I help you with, Arthur?', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
  HumanMessage(content='what is the weather in sf?', additional_kwargs={}, response_metadata={}),
  AIMessage(content="Right now in San Francisco, it's about **15.3 °C** (roughly 59.5 °F). Classic SF — mild, probably a little f